# MOE

In [2]:
# part 1: 导入相关的 package
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from dataclasses import dataclass

import math

torch.manual_seed(1024)

# Export Mask

In [14]:
# 形状是 n_export * n_top * (seq_len * batch_size)
n_export = 5
n_top = 3
seq_len = 3
batch_size = 2

export_mask =torch.tril( torch.ones((n_export, n_top, seq_len * batch_size)))
export_mask

tensor([[[1., 0., 0., 0., 0., 0.],
         [1., 1., 0., 0., 0., 0.],
         [1., 1., 1., 0., 0., 0.]],

        [[1., 0., 0., 0., 0., 0.],
         [1., 1., 0., 0., 0., 0.],
         [1., 1., 1., 0., 0., 0.]],

        [[1., 0., 0., 0., 0., 0.],
         [1., 1., 0., 0., 0., 0.],
         [1., 1., 1., 0., 0., 0.]],

        [[1., 0., 0., 0., 0., 0.],
         [1., 1., 0., 0., 0., 0.],
         [1., 1., 1., 0., 0., 0.]],

        [[1., 0., 0., 0., 0., 0.],
         [1., 1., 0., 0., 0., 0.],
         [1., 1., 1., 0., 0., 0.]]])

In [18]:
for i in range(n_export):
    idx,top_x = torch.where(export_mask[i])
    print(idx) # 行索引 (对应top几） 
    print(top_x)  # 列索引 （对应第几个token)
    # 行索引和列索引 对应这个export_mask的意义是什么
    # idx_t：被第i个专家选中的第t个token是top几
    # top_x_t： 被第i个专家选中的第t个token的在原始序列中的token序号
    
    # 我有这两个变量有什么用
    
    break

tensor([0, 1, 1, 2, 2, 2])
tensor([0, 0, 1, 0, 1, 2])


# Sparse MOE 

In [ ]:
class ModelConfig:
    n_export: int = 8
    n_top: int = 4
    seq_len: int = 1024
    batch_size: int = 2
    hidden_dim: int = 512

In [ ]:
class SingleExport(nn.Module):
    def __init__(self, input_dim, output_dim) -> None:
        super().__init__()
        self.fn = nn.Linear(input_dim,output_dim)
    def forward(self, x):
        return self.fn(x)

In [ ]:
class MoeRouter(nn.Module):
    def __init__(self, n_export, hidden_dim, k) -> None:
        super().__init__()
        self.gate = nn.Linear(hidden_dim, n_export)
        self.n_export = n_export
        self.k = k

    def forward(self, x: torch.Tensor):

        route_logits = self.gate(x)

        route_porb = F.softmax(route_logits, dim=-1)
        route_weight, selected_export = torch.topk(route_porb, k=self.k, dim=-1)

        route_weight = route_weight / route_weight.sum(dim=-1, keepdim=True)

        route_weight = route_weight.to(x.dtype)

        mask_export = F.one_hot(route_weight, num_classes=self.n_export)
        mask_export = mask_export.permute(2, 1, 0)

        return route_weight, selected_export, mask_export


class SparseMoe(nn.Module):
    def __init__(self, config: ModelConfig) -> None:
        super().__init__()

        self.export_list = nn.ModuleList(
            [SingleExport(config.hidden_dim, config.hidden_dim) for _ in range(config.n_export)]
        )

        self.router = MoeRouter(config.n_export, config.hidden_dim, config.n_top)
        
        self.n_export = config.n_export
        
    def forward(self,hidden_state):
        # hidden_state shepe: (batch_size, seq_len, hidden_dim)
        batch_size, seq_len, hidden_dim = hidden_state.szie()
        hidden_state = hidden_state.view(-1,hiddem_dim)  # shape: (batch_size * seq_len, hidden_dim)
        
        route_weight, selected_export, mask_export = self.router(hidden_state)
        
        final_state = torch.zeros(batch_size*seq_len,hidden_dim)
        
        for export_idx in range(self.n_export):
            export_layer = self.export_list[export_idx]
            current_mask_export = mask_export[epxort_idx]
            
            top_idx, top_x=torch.where(current_mask_export)  
            # top_idx是 export_idx这个专家 选中的token 他是top几，
            # top_x 是 export_idx这个专家 选中的token 他的序号是几
            
            current_state = export_layer(hidden_state[top_x,:])
            
            current_route_weight = route_weight[top_x,top_idx].unsqueeze(-1) * current_state
            final_state.index(a)
            
            
        
            

In [11]:
torch.zeros(1,5).unsqueeze(-1)

tensor([[[0.],
         [0.],
         [0.],
         [0.],
         [0.]]])

In [5]:
test_x = torch.randn((2,3,4))
print(test_x.shape)
test_x.view(-1,4).shape

torch.Size([2, 3, 4])


torch.Size([6, 4])

In [9]:
test_x = torch.tensor([[1,0,1],[1,1,0]])
print(test_x.size())
a,b=test_x.size()
torch.where(test_x,)

torch.Size([2, 3])


(tensor([0, 0, 1, 1]), tensor([0, 2, 0, 1]))


## ExportRouter

In [ ]:
class SingleExpert(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        
    def forward(self, x):
        return self.linear(x)
    

class ExportRouter(nn.Module):
    def __init__(self, n_export, n_top):
        super().__init__()
        self.router = nn.Linear(hidden_dim, n_export)
        self.n_top = n_top
        self.n_export = n_export
        
    def forward(self, x):
        
        export_logits = self.router(x)  # (seq_len * batch_size, n_export)
        
        export_probs = F.softmax(export_logits, dim=-1)  # (seq_len * batch_size, n_export)
        
        router_weight, export_idx = torch.topk(export_probs, self.n_top, dim=-1)  # (seq_len * batch_size, n_top)
        
        masked_export = one_hot(export_idx, self.n_export)  # (seq_len * batch_size, n_top, n_export)
        
        masked_export = masked_export.permute(2,1,0)  # (n_export, n_top, seq_len * batch_size)
        return export_logits, router_weight, export_idx, masked_export
    
    
        

## SparseMOE

In [ ]:
class SparseMOE(nn.Module):
    def __init__(self, config):
        
        self.exports = nn.ModuleList([SingleExpert(config.hidden_dim, config.hidden_dim) for _ in range(config.n_export)])
        
        self.router = ExportRouter(config.n_export, config.n_top)
        
    def forward(self, x):
        
        seq_len, batch_size, hidden_dim = x.shape
        
        x = x.view(seq_len * batch_size, hidden_dim)  # (seq_len * batch_size, hidden_dim)
        
        final_hidden_state = torch.zeros(seq_len * batch_size, hidden_dim)
        
        export_logits, router_weight, export_idx, masked_export = self.router(x)
        export_x = self.exports[export_idx](x)  
        for export_idx in range(config.n_export):
            
            idx, top_x = torch.where(masked_export[export_idx])
            
            
            
            current_state = router_weight[top_x, idx] * export_x[top_x]
            
            final_hidden_state.add_index_(0,top_x, router_weight_i)
            
        
        return final_hidden_state.view(seq_len, batch_size, hidden_dim)
        